# **Layer Normalization (LayerNorm)**

## 1. What is Layer Normalization?

Layer Normalization stabilizes and accelerates the training of deep neural networks by normalizing the inputs **across the features/hidden units of a single data sample**, rather than across the entire mini-batch.

### **The Core Intuition: Individual Check vs. Group Curve**
* **Batch Normalization (Group Grading):** Grades a student by comparing them against everyone else in the entire class (across the batch). If the class size is small or changes, the grade swings wildly.
* **Layer Normalization (Self-Calibration):** Calibrates each student based purely on their own personal set of scores (across all features of that single sample). It does not care about what anyone else in the batch is doing.

## 2. Why Did We Need LayerNorm? (The Flaws of BatchNorm)

Before LayerNorm, **Batch Normalization (BatchNorm)** was king in Computer Vision. However, **BatchNorm completely fails when applied to sequential models (RNNs/LSTMs) and Transformers** for two major reasons:

### **Flaw 1: Variable-Length Sequences in NLP**
In NLP, sentences in a batch have different lengths (e.g., Sentence 1 has 5 words, Sentence 2 has 20 words). 
* BatchNorm requires computing the mean and variance across the batch at each time step.
* For late time steps (e.g., word 18), only a few sentences are still active, making the batch statistics noisy, erratic, and unstable.
    - If a sentence has a less words, so to keep the input size same for all the input matrixes it uses padding (0 padding) for the inputs which has less words, which creates lot of sparcity in the data.

### **Flaw 2: Batch Size Dependency**
* BatchNorm depends heavily on having a reasonably large batch size (e.g., 32, 64) to compute accurate statistics.
* If the batch size is 1 (e.g., during real-time online inference), BatchNorm cannot compute variance.
* In massive LLM training where GPU memory only allows tiny micro-batches (e.g., batch size 1 or 2 per GPU), BatchNorm breaks down completely.

> **LayerNorm solves both problems:** Because it normalizes across features of each individual token/sample independently, it works identically regardless of batch size (even with batch size = 1) and handles variable sequence lengths effortlessly.

## 3. BatchNorm vs. LayerNorm: Dimensional Comparison

Consider a 2D activation matrix of shape `(Batch Size, Feature Dimension)`:

```text
                 Features (Dimensions d_model)
              f1      f2      f3      f4
        ┌──────────────────────────────────┐
Sample 1│  x11     x12     x13     x14     │  ──► LayerNorm: Normalizes across ROW (Sample 1)
Sample 2│  x21     x22     x23     x24     │  ──► LayerNorm: Normalizes across ROW (Sample 2)
Sample 3│  x31     x32     x33     x34     │  ──► LayerNorm: Normalizes across ROW (Sample 3)
        └──────────────────────────────────┘
            ▲       ▲       ▲       ▲
            │       │       │       │
            BatchNorm normalizes down each COLUMN (across the Batch)
```

| Feature | Batch Normalization (BatchNorm) | Layer Normalization (LayerNorm) |
| :--- | :--- | :--- |
| **Direction of Normalization** | Across the **batch** (vertically) | Across the **features** (horizontally) |
| **Dependencies** | Depends on other samples in the batch | 100% independent per sample |
| **Works with Batch Size = 1?** | ❌ No (variance is 0) | ✅ Yes (fully supported) |
| **Training vs. Inference** | Needs running mean/var for testing | Uses the exact same formula for train & test |
| **Primary Use Case** | Computer Vision (CNNs, ConvNets) | NLP, Transformers, LLMs, RNNs |

## 4. The Mathematical Formulation

For a given input vector $x = [x_1, x_2, \dots, x_n]$ with $n$ hidden features:

### **Step 1: Compute Mean ($\mu$)**
$$\mu = \frac{1}{n} \sum_{i=1}^{n} x_i$$

### **Step 2: Compute Variance ($\sigma^2$)**
$$\sigma^2 = \frac{1}{n} \sum_{i=1}^{n} (x_i - \mu)^2$$

### **Step 3: Normalize to Zero Mean and Unit Variance ($\hat{x}_i$)**
$$\hat{x}_i = \frac{x_i - \mu}{\sqrt{\sigma^2 + \epsilon}}$$

### **Step 4: Scale and Shift (Affine Transformation)**
$$y_i = \gamma \cdot \hat{x}_i + \beta$$

### **Where,**
* **$n$:** Number of features / hidden units (e.g., $d_{\text{model}} = 512$ in Transformers).
* **$\mu$:** The mean of the features for this specific sample.
* **$\sigma^2$:** The variance of the features for this specific sample.
* **$\epsilon$ (Epsilon):** A tiny constant (e.g., $10^{-5}$) added to prevent division by zero.
* **$\hat{x}_i$:** The standardized value (mean = 0, variance = 1).
* **$\gamma$ (Gamma):** Learnable scale parameter (initialized to $1$).
* **$\beta$ (Beta):** Learnable shift parameter (initialized to $0$).

> **Why do we need $\gamma$ and $\beta$ ?**<br>
> If forcing mean = 0 and variance = 1 hurts the representational capacity of a layer, the neural network can learn to adjust $\gamma$ and $\beta$ during backpropagation to restore whatever scale and shift it needs.

## 5. Step-by-Step Mathematical Walkthrough

Let's trace a concrete numerical example with a single word token represented by **$4$ features** ($n = 4$):

$$x = [2.0, \; 4.0, \; 6.0, \; 8.0]$$

Assume learnable parameters: $\gamma = 1.0$, $\beta = 0.0$, and $\epsilon = 0.0$.

### **Step 1: Compute Mean ($\mu$)**
$$\mu = \frac{2.0 + 4.0 + 6.0 + 8.0}{4} = \frac{20.0}{4} = \mathbf{5.0}$$

### **Step 2: Compute Variance ($\sigma^2$)**
$$\sigma^2 = \frac{(2-5)^2 + (4-5)^2 + (6-5)^2 + (8-5)^2}{4}$$
$$\sigma^2 = \frac{(-3)^2 + (-1)^2 + (1)^2 + (3)^2}{4} = \frac{9 + 1 + 1 + 9}{4} = \frac{20}{4} = \mathbf{5.0}$$

Standard Deviation $\sigma = \sqrt{5.0} \approx \mathbf{2.236}$

### **Step 3: Normalize Each Element ($\hat{x}_i$)**
* $\hat{x}_1 = \frac{2.0 - 5.0}{2.236} = \frac{-3.0}{2.236} \approx \mathbf{-1.341}$
* $\hat{x}_2 = \frac{4.0 - 5.0}{2.236} = \frac{-1.0}{2.236} \approx \mathbf{-0.447}$
* $\hat{x}_3 = \frac{6.0 - 5.0}{2.236} = \frac{1.0}{2.236} \approx \mathbf{+0.447}$
* $\hat{x}_4 = \frac{8.0 - 5.0}{2.236} = \frac{3.0}{2.236} \approx \mathbf{+1.341}$

Normalized Vector: $\mathbf{\hat{x} = [-1.341, \; -0.447, \; 0.447, \; 1.341]}$

*(Notice: The sum of elements is 0, meaning mean = 0, and variance = 1!)*

### **Step 4: Scale and Shift ($y = \gamma \hat{x} + \beta$)**
Since $\gamma = 1.0$ and $\beta = 0.0$:
$$y = 1.0 \cdot \hat{x} + 0.0 = \mathbf{[-1.341, \; -0.447, \; 0.447, \; 1.341]}$$

## 6. LayerNorm in Transformers: Pre-LN vs. Post-LN

In the original 2017 Transformer paper ("Attention Is All You Need"), LayerNorm was applied **after** the residual addition. This is called **Post-LN**.

### **1. Post-LN (Original 2017 Transformer)**
$$x_{l+1} = \text{LayerNorm}(x_l + \text{Sublayer}(x_l))$$
* **Drawback:** Gradients can explode or vanish in the early layers of very deep networks without a strict learning rate warmup schedule.

### **2. Pre-LN (Modern Standard: GPT-3, LLaMA, Mistral)**
Modern architectures apply LayerNorm **before** the sub-layer and pass the residual connection clean and untouched:
$$x_{l+1} = x_l + \text{Sublayer}(\text{LayerNorm}(x_l))$$
* **Advantage:** Preserves an unimpeded residual stream directly from input to output, making training much more stable and eliminating the need for delicate warmups.

### **3. Modern Evolution: RMSNorm (Root Mean Square Normalization)**
Models like **LLaMA and Mistral** replace standard LayerNorm with **RMSNorm**:
* Drops the mean calculation ($\mu$) and only scales by the root mean square of inputs.
* Achieves identical stability while being **10% to 50% faster** to compute on GPUs!

## 7. Implementation: From Scratch & Frameworks

### **A. NumPy Implementation From Scratch**

```python
import numpy as np

def layer_norm(x, gamma=1.0, beta=0.0, eps=1e-5):
    # Compute mean and variance along the last axis (features)
    mean = np.mean(x, axis=-1, keepdims=True)
    var = np.var(x, axis=-1, keepdims=True)
    
    # Normalize
    x_norm = (x - mean) / np.sqrt(var + eps)
    
    # Scale and Shift
    return gamma * x_norm + beta

# Test with our sample
x_sample = np.array([[2.0, 4.0, 6.0, 8.0]])
print("LayerNorm Output:\n", layer_norm(x_sample))
```

---

### **B. TensorFlow / Keras Implementation**

```python
import tensorflow as tf
from tensorflow.keras import layers

# Create LayerNormalization layer
ln_layer = layers.LayerNormalization(epsilon=1e-5)

# Input tensor: (batch_size=2, seq_len=3, d_model=4)
inputs = tf.random.normal((2, 3, 4))
normalized_output = ln_layer(inputs)

print("Output shape:", normalized_output.shape)
```

---

### **C. PyTorch Implementation**

```python
import torch
import torch.nn as nn

# Define LayerNorm for embedding dimension of 512
layer_norm = nn.LayerNorm(normalized_shape=512)

# Input tensor of shape (batch_size=32, seq_len=10, d_model=512)
x = torch.randn(32, 10, 512)
output = layer_norm(x)
print("PyTorch Output shape:", output.shape)
```